
## Folder 06 — Referential Integrity Validation


**Referential integrity** ensures that `records in one table reference records` that actually `exist in another table`.


For this e-commerce project, the main relationships are:

- **Orders → Customers**
- **Order Items → Orders**
- **Order Items → Products**
- **Payments → Orders**
- **Shipments → Orders**
- **Inventory → Products**

For example, if an order references CUST99999, but that customer does not exist in the customers table, that order has an orphan customer reference.

This notebook detects those problems and stores the results in **Delta tables**.


### Referential Integrity Architecture


```text
┌──────────────────────────────────────────────┐
│              SILVER DELTA TABLES             │
│                                              │
│  Customers · Products · Orders               │
│  Order Items · Payments · Shipments           │
│  Inventory                                   │
└──────────────────────────────────────────────┘
                       │
                       ▼
┌──────────────────────────────────────────────┐
│         REFERENTIAL INTEGRITY VALIDATION     │
│                                              │
│  • Validate Foreign Key References           │
│  • Match Source FK with Target Key           │
│  • Identify Missing Parent Records           │
└──────────────────────────────────────────────┘
                       │
              ┌────────┴────────┐
              ▼                 ▼
┌──────────────────────┐  ┌──────────────────────┐
│  RELATIONSHIP REPORT │  │   ORPHAN RECORDS     │
│                      │  │                      │
│  Validation Results  │  │  Missing References │
│  Orphan Counts       │  │  Source Record Data │
│  PASS / FAIL Status  │  │  Relationship Name  │
└──────────────────────┘  └──────────────────────┘
```

In [0]:
# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

from pyspark.sql import functions as F
from datetime import datetime, timezone

print("Libraries imported successfully.")


# ============================================================
# 2. PROJECT CONFIGURATION
# ============================================================

CATALOG = "workspace"
SCHEMA = "ecommerce_silver"

REPORT_TABLE = (
    f"{CATALOG}.{SCHEMA}.referential_integrity_report"
)

ORPHAN_TABLE = (
    f"{CATALOG}.{SCHEMA}.referential_integrity_orphans"
)

RUN_ID = spark.sql(
    "SELECT uuid() AS run_id"
).first()["run_id"]

print("Run ID:", RUN_ID)


# ============================================================
# 3. SILVER TABLE CONFIGURATION
# ============================================================

TABLES = {
    "customers": f"{CATALOG}.{SCHEMA}.silver_customers",
    "products": f"{CATALOG}.{SCHEMA}.silver_products",
    "orders": f"{CATALOG}.{SCHEMA}.silver_orders",
    "order_items": f"{CATALOG}.{SCHEMA}.silver_order_items",
    "payments": f"{CATALOG}.{SCHEMA}.silver_payments",
    "shipments": f"{CATALOG}.{SCHEMA}.silver_shipments",
    "inventory": f"{CATALOG}.{SCHEMA}.silver_inventory"
}


# ============================================================
# 4. LOAD AND VALIDATE SILVER TABLES
# ============================================================

silver_dataframes = {}

for dataset_name, table_name in TABLES.items():

    if not spark.catalog.tableExists(table_name):

        raise Exception(
            f"Required Silver table does not exist: {table_name}"
        )

    loaded_df = spark.table(table_name)

    silver_dataframes[dataset_name] = loaded_df

    print(
        f"Loaded: {table_name} | "
        f"Columns: {len(loaded_df.columns)}"
    )

print("-" * 70)
print("All Silver tables loaded successfully.")


# ============================================================
# 5. DEFINE FOREIGN-KEY RELATIONSHIPS
# ============================================================

RELATIONSHIPS = [

    {
        "relationship": "orders_to_customers",
        "source_dataset": "orders",
        "source_fk": "customer_id",
        "source_pk": "order_id",
        "target_dataset": "customers",
        "target_key": "customer_id"
    },

    {
        "relationship": "order_items_to_orders",
        "source_dataset": "order_items",
        "source_fk": "order_id",
        "source_pk": "order_item_id",
        "target_dataset": "orders",
        "target_key": "order_id"
    },

    {
        "relationship": "order_items_to_products",
        "source_dataset": "order_items",
        "source_fk": "product_id",
        "source_pk": "order_item_id",
        "target_dataset": "products",
        "target_key": "product_id"
    },

    {
        "relationship": "payments_to_orders",
        "source_dataset": "payments",
        "source_fk": "order_id",
        "source_pk": "payment_id",
        "target_dataset": "orders",
        "target_key": "order_id"
    },

    {
        "relationship": "shipments_to_orders",
        "source_dataset": "shipments",
        "source_fk": "order_id",
        "source_pk": "shipment_id",
        "target_dataset": "orders",
        "target_key": "order_id"
    },

    {
        "relationship": "inventory_to_products",
        "source_dataset": "inventory",
        "source_fk": "product_id",
        "source_pk": "inventory_snapshot_id",
        "target_dataset": "products",
        "target_key": "product_id"
    }
]


# ============================================================
# 6. DEFINE HELPER FUNCTION
# ============================================================

def is_blank(column_expression):

    return (
        column_expression.isNull()
        | (
            F.trim(
                column_expression.cast("string")
            ) == ""
        )
    )


# ============================================================
# 7. DEFINE REFERENTIAL INTEGRITY VALIDATION FUNCTION
# ============================================================

def validate_relationship(config, source_data_df, target_data_df):

    relationship = config["relationship"]

    source_dataset = config["source_dataset"]
    source_fk = config["source_fk"]
    source_pk = config["source_pk"]

    target_dataset = config["target_dataset"]
    target_key = config["target_key"]

    print(
        f"Validating relationship: {relationship}"
    )

    # --------------------------------------------------------
    # 7.1 Validate required columns
    # --------------------------------------------------------

    required_source_columns = [
        source_fk,
        source_pk
    ]

    for column_name in required_source_columns:

        if column_name not in source_data_df.columns:

            raise Exception(
                f"Missing source column: "
                f"{source_dataset}.{column_name}"
            )

    if target_key not in target_data_df.columns:

        raise Exception(
            f"Missing target key: "
            f"{target_dataset}.{target_key}"
        )

    # --------------------------------------------------------
    # 7.2 Count source and target records
    # --------------------------------------------------------

    source_count = source_data_df.count()

    target_count = target_data_df.count()

    # --------------------------------------------------------
    # 7.3 Identify null and blank foreign keys
    # --------------------------------------------------------

    null_fk_data_df = source_data_df.filter(
        is_blank(F.col(source_fk))
    )

    null_fk_count = null_fk_data_df.count()

    # --------------------------------------------------------
    # 7.4 Identify duplicate target keys
    # --------------------------------------------------------

    duplicate_target_keys_df = (
        target_data_df
        .filter(
            ~is_blank(F.col(target_key))
        )
        .groupBy(target_key)
        .count()
        .filter(
            F.col("count") > 1
        )
    )

    duplicate_target_key_groups = (
        duplicate_target_keys_df.count()
    )

    # --------------------------------------------------------
    # 7.5 Build distinct target reference lookup
    # --------------------------------------------------------

    target_reference_df = (
        target_data_df
        .select(
            F.trim(
                F.col(target_key).cast("string")
            ).alias("_reference_key")
        )
        .filter(
            F.col("_reference_key").isNotNull()
            & (F.col("_reference_key") != "")
        )
        .distinct()
    )

    # --------------------------------------------------------
    # 7.6 Exclude null and blank source foreign keys
    # --------------------------------------------------------

    valid_source_data_df = (
        source_data_df
        .filter(
            ~is_blank(F.col(source_fk))
        )
        .withColumn(
            "_source_reference_key",
            F.trim(
                F.col(source_fk).cast("string")
            )
        )
    )

    # --------------------------------------------------------
    # 7.7 Identify orphan records using LEFT ANTI JOIN
    # --------------------------------------------------------

    orphan_data_df = (
        valid_source_data_df.alias("source")
        .join(
            target_reference_df.alias("target"),
            F.col("source._source_reference_key")
            == F.col("target._reference_key"),
            "left_anti"
        )
        .drop("_source_reference_key")
    )

    orphan_count = orphan_data_df.count()

    # --------------------------------------------------------
    # 7.8 Prepare common invalid-record structure
    # --------------------------------------------------------

    source_columns = source_data_df.columns

    source_record_json = F.to_json(
        F.struct(
            *[
                F.col(column_name)
                for column_name in source_columns
            ]
        )
    )

    # --------------------------------------------------------
    # 7.9 Prepare orphan record details
    # --------------------------------------------------------

    orphan_detail_data_df = (
        orphan_data_df
        .select(
            F.lit(relationship).alias("relationship"),

            F.lit(source_dataset).alias("source_dataset"),

            F.lit(source_fk).alias("foreign_key_column"),

            F.lit(target_dataset).alias("target_dataset"),

            F.lit(target_key).alias("target_key_column"),

            F.col(source_pk).cast("string").alias(
                "source_record_id"
            ),

            F.col(source_fk).cast("string").alias(
                "invalid_reference_value"
            ),

            source_record_json.alias(
                "source_record_json"
            ),

            F.lit(RUN_ID).alias("run_id"),

            F.current_timestamp().alias("detected_at")
        )
    )

    # --------------------------------------------------------
    # 7.10 Prepare null foreign-key record details
    # --------------------------------------------------------

    null_fk_detail_data_df = (
        null_fk_data_df
        .select(
            F.lit(relationship).alias("relationship"),

            F.lit(source_dataset).alias("source_dataset"),

            F.lit(source_fk).alias("foreign_key_column"),

            F.lit(target_dataset).alias("target_dataset"),

            F.lit(target_key).alias("target_key_column"),

            F.col(source_pk).cast("string").alias(
                "source_record_id"
            ),

            F.col(source_fk).cast("string").alias(
                "invalid_reference_value"
            ),

            source_record_json.alias(
                "source_record_json"
            ),

            F.lit(RUN_ID).alias("run_id"),

            F.current_timestamp().alias("detected_at")
        )
    )

    # --------------------------------------------------------
    # 7.11 Combine orphan and null-reference records
    # --------------------------------------------------------

    invalid_reference_data_df = (
        orphan_detail_data_df
        .unionByName(
            null_fk_detail_data_df
        )
    )

    # --------------------------------------------------------
    # 7.12 Determine relationship status
    # --------------------------------------------------------

    relationship_status = (
        "PASS"
        if (
            null_fk_count == 0
            and orphan_count == 0
            and duplicate_target_key_groups == 0
        )
        else "FAIL"
    )

    # --------------------------------------------------------
    # 7.13 Build relationship result
    # --------------------------------------------------------

    result = {

        "relationship": relationship,

        "source_dataset": source_dataset,

        "source_foreign_key": source_fk,

        "target_dataset": target_dataset,

        "target_key": target_key,

        "source_records": source_count,

        "target_records": target_count,

        "null_foreign_key_records": null_fk_count,

        "orphan_records": orphan_count,

        "duplicate_target_key_groups": (
            duplicate_target_key_groups
        ),

        "status": relationship_status,

        "run_id": RUN_ID,

        "checked_at": datetime.now(timezone.utc).replace(
            tzinfo=None
        )
    }

    print(
        f"Completed: {relationship} | "
        f"Status: {relationship_status}"
    )

    return result, invalid_reference_data_df


# ============================================================
# 8. EXECUTE ALL RELATIONSHIP CHECKS
# ============================================================

validation_results = []

orphan_dataframes = []

for relationship_config in RELATIONSHIPS:

    source_name = relationship_config["source_dataset"]

    target_name = relationship_config["target_dataset"]

    source_input_df = silver_dataframes[source_name]

    target_input_df = silver_dataframes[target_name]

    result, invalid_records_df = validate_relationship(
        relationship_config,
        source_input_df,
        target_input_df
    )

    validation_results.append(result)

    orphan_dataframes.append(invalid_records_df)

    print("Source records:", result["source_records"])

    print(
        "Null foreign keys:",
        result["null_foreign_key_records"]
    )

    print(
        "Orphan records:",
        result["orphan_records"]
    )

    print(
        "Duplicate target key groups:",
        result["duplicate_target_key_groups"]
    )

    print("Status:", result["status"])

    print("-" * 70)


# ============================================================
# 9. CREATE RELATIONSHIP VALIDATION REPORT
# ============================================================

report_df = spark.createDataFrame(
    validation_results
)

print("RELATIONSHIP VALIDATION REPORT")

display(
    report_df.orderBy("relationship")
)


# ============================================================
# 10. COMBINE ALL INVALID REFERENCE RECORDS
# ============================================================

if len(orphan_dataframes) == 0:

    raise Exception(
        "No relationship validation results were generated."
    )

combined_orphans_df = orphan_dataframes[0]

for next_orphans_df in orphan_dataframes[1:]:

    combined_orphans_df = (
        combined_orphans_df.unionByName(
            next_orphans_df
        )
    )

print("Combined invalid reference records:")

display(combined_orphans_df)


# ============================================================
# 11. SAVE REFERENTIAL INTEGRITY REPORT
# ============================================================

(
    report_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(REPORT_TABLE)
)

print(
    "Referential integrity report saved:",
    REPORT_TABLE
)


# ============================================================
# 12. SAVE ORPHAN AND NULL-REFERENCE RECORDS
# ============================================================

(
    combined_orphans_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(ORPHAN_TABLE)
)

print(
    "Orphan records saved:",
    ORPHAN_TABLE
)


# ============================================================
# 13. VERIFY SAVED REPORTS
# ============================================================

saved_report_df = spark.table(REPORT_TABLE)

saved_orphans_df = spark.table(ORPHAN_TABLE)

saved_report_count = saved_report_df.count()

saved_orphan_count = saved_orphans_df.count()

if saved_report_count != len(RELATIONSHIPS):

    raise Exception(
        "Referential integrity report verification failed. "
        f"Expected {len(RELATIONSHIPS)} rows, "
        f"found {saved_report_count}."
    )

print("Report verification successful.")

print("Saved report rows:", saved_report_count)

print("Saved invalid-reference rows:", saved_orphan_count)


# ============================================================
# 14. RELATIONSHIP-LEVEL SUMMARY
# ============================================================

summary_df = (
    saved_report_df
    .agg(

        F.count("*").alias(
            "total_relationships"
        ),

        F.sum(
            F.when(
                F.col("status") == "PASS",
                1
            ).otherwise(0)
        ).alias(
            "passed_relationships"
        ),

        F.sum(
            F.when(
                F.col("status") == "FAIL",
                1
            ).otherwise(0)
        ).alias(
            "failed_relationships"
        ),

        F.sum(
            "null_foreign_key_records"
        ).alias(
            "total_null_foreign_keys"
        ),

        F.sum(
            "orphan_records"
        ).alias(
            "total_orphan_records"
        ),

        F.sum(
            "duplicate_target_key_groups"
        ).alias(
            "total_duplicate_target_key_groups"
        )
    )
    .withColumn(
        "overall_status",
        F.when(
            F.col("failed_relationships") == 0,
            "PASS"
        ).otherwise("FAIL")
    )
)

print("REFERENTIAL INTEGRITY SUMMARY")

display(summary_df)


# ============================================================
# 15. INSPECT INVALID REFERENCE RECORDS
# ============================================================

print("Orphan and null-reference records:")

display(
    saved_orphans_df
    .orderBy(
        "relationship",
        "invalid_reference_value"
    )
)


# ============================================================
# 16. INSPECT DUPLICATE PARENT KEYS
# ============================================================

for relationship_config in RELATIONSHIPS:

    target_name = relationship_config["target_dataset"]

    target_key_name = relationship_config["target_key"]

    target_input_df = silver_dataframes[target_name]

    duplicate_keys_df = (
        target_input_df
        .filter(
            ~is_blank(F.col(target_key_name))
        )
        .groupBy(target_key_name)
        .count()
        .filter(
            F.col("count") > 1
        )
    )

    duplicate_count = duplicate_keys_df.count()

    if duplicate_count > 0:

        print(
            f"Duplicate keys found in "
            f"{TABLES[target_name]}.{target_key_name}"
        )

        display(duplicate_keys_df)

    else:

        print(
            f"No duplicate parent keys: "
            f"{target_name}.{target_key_name}"
        )


# ============================================================
# 17. FINAL VALIDATION SUMMARY
# ============================================================

total_relationships = saved_report_df.count()

passed_relationships = (
    saved_report_df
    .filter(
        F.col("status") == "PASS"
    )
    .count()
)

failed_relationships = (
    saved_report_df
    .filter(
        F.col("status") == "FAIL"
    )
    .count()
)

total_null_foreign_keys = (
    saved_report_df
    .agg(
        F.sum("null_foreign_key_records")
    )
    .first()[0] or 0
)

total_orphans = (
    saved_report_df
    .agg(
        F.sum("orphan_records")
    )
    .first()[0] or 0
)

total_duplicate_groups = (
    saved_report_df
    .agg(
        F.sum("duplicate_target_key_groups")
    )
    .first()[0] or 0
)

print("=" * 70)

print("REFERENTIAL INTEGRITY VALIDATION SUMMARY")

print("=" * 70)

print("Run ID:", RUN_ID)

print("Total relationships:", total_relationships)

print("Passed relationships:", passed_relationships)

print("Failed relationships:", failed_relationships)

print("Null foreign-key records:", total_null_foreign_keys)

print("Orphan records:", total_orphans)

print("Duplicate target-key groups:", total_duplicate_groups)

print("Saved orphan records:", saved_orphan_count)

print("=" * 70)

if failed_relationships > 0:

    print(
        "WARNING: Referential integrity issues detected."
    )

    print(
        "Review the report and orphan tables."
    )

else:

    print(
        "All configured relationships passed."
    )

print("Report table:", REPORT_TABLE)

print("Orphan table:", ORPHAN_TABLE)

print("=" * 70)


# ============================================================
# 18. NOTEBOOK EXECUTION RESULT
# ============================================================

print(
    f"SUCCESS | RUN_ID={RUN_ID} | "
    f"RELATIONSHIPS={total_relationships} | "
    f"PASSED={passed_relationships} | "
    f"FAILED={failed_relationships} | "
    f"ORPHANS={total_orphans}"
)

dbutils.notebook.exit(
    f"SUCCESS|RUN_ID={RUN_ID}|"
    f"RELATIONSHIPS={total_relationships}|"
    f"PASSED={passed_relationships}|"
    f"FAILED={failed_relationships}|"
    f"ORPHANS={total_orphans}"
)

In [0]:
%sql
SELECT
    relationship,
    COUNT(*) AS invalid_record_count
FROM workspace.ecommerce_silver.referential_integrity_orphans
GROUP BY relationship
ORDER BY relationship;


## 2. Expected Relationship Report


*Illustrative output — actual counts depend on your Silver Delta tables.*

| Relationship | Source Foreign Key (FK) | Target Primary/Business Key |
|---|---|---|
| Orders → Customers | `customer_id` | `customer_id` |
| Order Items → Orders | `order_id` | `order_id` |
| Order Items → Products | `product_id` | `product_id` |
| Payments → Orders | `order_id` | `order_id` |
| Shipments → Orders | `order_id` | `order_id` |
| Inventory → Products | `product_id` | `product_id` |


### Referential Integrity Report Tables


Two Delta tables are created to store validation results.

| Delta Table | Purpose |
|---|---|
| `referential_integrity_report` | Stores relationship-wise validation results, orphan counts, and validation status. |
| `referential_integrity_orphans` | Stores individual orphan records with missing parent references. |


### Why use a LEFT ANTI JOIN?

In [0]:
orphan_df = (
    source_df
    .join(
        target_df,
        source_df.customer_id == target_df.customer_id,
        "left_anti"
    )
)


A **left anti join** returns `source records` that have **no matching record** in the **target table**.


### Why check duplicate target keys?


- If the customers table contains two rows with the same customer_id, joining orders to customers could multiply order records. 
- A parent key should be unique for a conventional one-to-many relationship.